<a href="https://colab.research.google.com/github/dilsh9810/IT3437_Assignment02/blob/master/IT5437_Assignment02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. **Line Fitting**

**a**) **Total** **Least** **Squares** **(TLS)**

The Orthoginal distance in Total Least Square for the first line in the dataset(x1,y1) is calculated based on the orthogonal distances of the points from the fitted line. This process involves the application of Singular Value Decomposition after centering the data.


In [16]:
import numpy as np
import pandas as pd

# 1. Load the data
df = pd.read_csv('/content/sample_data/lines.csv')
# Clean column names by removing leading '#' and stripping whitespace
df.columns = df.columns.str.replace('# ', '', regex=False).str.strip()
x = df['x1'].values
y = df['y1'].values

# 2. Center the data
x_mean, y_mean = np.mean(x), np.mean(y)
pts_centered = np.vstack([x - x_mean, y - y_mean]).T

# 3. Perform SVD
# The last row of Vh (or last column of V) corresponds to
# the smallest singular value and represents the line normal.
_, _, vh = np.linalg.svd(pts_centered)
a, b = vh[1, :]

# 4. Calculate the intercept
c = -(a * x_mean + b * y_mean)

print(f"Resulting Line: {a:.4f}x + {b:.4f}y + {c:.4f} = 0")

Resulting Line: 0.7736x + -0.6337y + -3.7942 = 0


b). **Interative** **RANSAC**

In [14]:
from os.path import normcase
import numpy as np

data = np.genfromtxt("/content/sample_data/lines.csv", delimiter=",", skip_header=1)

x_value = data[:, :3]
y_value = data[:, 3:]

x_all = x_value.flatten()
y_all = y_value.flatten()

points = np.vstack([x_all, y_all]).T

#print(x_value)
#print(y_value)

#print(points)

def get_line_params(p1,p2):
    """Calculate normalized line parameters ax + by + d = 0. """
    v = p2 - p1
    n = np.array([-v[1], v[0]])
    norm = np.linalg.norm(n)
    print(norm)
    if norm == 0: return None
    n  = n / norm
    d = -np.dot(n, p1)
    return n[0], n[1], d


def ransac_line(pts, threshold=0.1, iterations=1000):
    best_params = None
    best_inliers = []

    for _ in range(iterations):

        # Pick 2 random points
        idx = np.random.choice(len(pts), 2, replace=False)
        p1, p2 = pts[idx]
        params = get_line_params(p1, p2)
        if params is None: continue

        a,b,d = params
        # Distance of all points to the line: |ax + by + d|
        dist = np.abs(a * pts[:, 0] + b * pts[:, 1] + d)
        inliers = np.where(dist < threshold)[0]

        if len(inliers) > len(best_inliers):
           best_inliers = inliers
           best_params  = params

        return best_params, best_inliers

# Iteratively find 3 lines
remaining_points = points.copy()
results = []

for i in range(3):
    params, inlier_indices = ransac_line(remaining_points)
    results.append(params)
    # Masking: Remove the found inliers from the dataset
    remaining_points = np.delete(remaining_points, inlier_indices, axis=0)
    print(f"Line {i+1} Parameters: {params[0]:.4f}x + {params[1]:.4f}y + {params[2]:.4f} = 0")



9.117305183298724
Line 1 Parameters: -0.9998x + 0.0180y + -4.1578 = 0
5.722892389636077
Line 2 Parameters: 0.0586x + -0.9983y + -3.1763 = 0
6.37034240632266
Line 3 Parameters: 0.7541x + -0.6568y + 0.6041 = 0
